# beatgen — Train.ipynb

Простой end-to-end ноутбук: чек окружения → сборка датасета → Stage 1 → Stage 2 → смок-инференс.

Под разное железо крути пресет в ячейке **Config** — там же список того, что вообще можно крутить (см. секцию «Что крутить» внизу).

## Что крутить (VRAM → настройки)

Двигай ползунок VRAM в следующей ячейке — он посчитает `--bs`, `--steps`, `--epochs`, `--lr` под твою карту. Stage 2 принимает только `--epochs` и `--lr` (там нет флагов `--bs`/`--steps`), так что основной рычаг ускорения на жирной карте — это Stage 1 + длиннее epochs.

Ориентиры:
- bf16 на Ampere+ (3060/4090/A100/H200) — быстрее и стабильнее.
- fp16 — для Volta/Turing.
- AMP off — только CPU или дебаг.
- `--lr` выше ~2e-3 для Adam расходится.
- Потолок F1 на 326 OST ≈ 0.56 — выше не прыгнуть без BeatSaver.

In [ ]:
# 1. Окружение
import os, sys, shutil, subprocess, json, platform
from pathlib import Path

ROOT = Path(r"E:\git\beatgen").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

PY = str(ROOT / ".venv" / "Scripts" / "python.exe")
print("CWD  :", os.getcwd())
print("Py   :", PY, "->", "ok" if Path(PY).exists() else "MISSING")
print("OS   :", platform.platform())

try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        cc = torch.cuda.get_device_capability(0)
        print(f"GPU  : {name}  VRAM={vram:.1f}GB  CC={cc}")
except Exception as e:
    print("torch import failed:", e)

In [ ]:
ui_vram  = w.IntSlider(value=12, min=0, max=160, step=1,
                       description="VRAM, ГБ", continuous_update=False)
ui_amp   = w.ToggleButtons(options=["bf16", "fp16", "off"], value="bf16",
                           description="AMP")
ui_out   = w.Output()

display(w.VBox([ui_vram, ui_amp, ui_out]))

# cfg живёт между ячейками — определяем как глобал, чтобы ячейки 4–6 читали его.
cfg = {}


def _render(_=None):
    cfg.clear()
    cfg.update(recommend(ui_vram.value, ui_amp.value))
    with ui_out:
        clear_output()
        print(json.dumps(cfg, indent=2, ensure_ascii=False))
        print()
        print("Подсказки:")
        print(" • stage2.py НЕ принимает --bs/--steps — bs тут влияет только на stage1.")
        print(" • --lr выше 2e-3 для Adam обычно расходится.")
        print(" • на H200 (141 GB) упираешься не в VRAM, а в I/O датасета — workers=24 потолок.")


# Пересчёт сразу при отпускании слайдера или смене AMP.
ui_vram.observe(_render, names="value")
ui_amp.observe(_render, names="value")
_render()

In [ ]:
# 3. Сборка датасета (если dataset/index.json ещё нет — пропусти если уже есть)
dataset_idx = Path(cfg["data"]) / "index.json"
if not dataset_idx.exists():
    # Скушает и OST-бандлы, и обычные папки кастомок (рекурсивно).
    src = ROOT / "BeatmapLevelsData"
    print("Building dataset from:", src)
    subprocess.check_call([PY, "extract/build_dataset.py", str(src)])
else:
    print("Dataset already exists ->", dataset_idx)
    n = len(json.loads(dataset_idx.read_text()).get("items", []))
    print("Songs in index:", n)

In [ ]:
# 4. Train Stage 1 (TCN onsets). Чекпойнты: models/_ckpt/stage1.latest.pt + .best.pt
cmd1 = [
    PY, "models/stage1.py",
    "--epochs", str(cfg["ep1"]),
    "--steps", str(cfg["steps1"]),
    "--bs",    str(cfg["bs"]),
    "--lr",    str(cfg["lr"]),
    "--device",cfg["device"],
    "--data",  cfg["data"],
    "--out-dir", cfg["out_dir"],
]
print(">>", " ".join(cmd1))
subprocess.check_call(cmd1)

In [ ]:
# 5. Train Stage 2 (GRU notes). NB: --bs/--steps не поддерживаются тут.
cmd2 = [
    PY, "models/stage2.py",
    "--epochs", str(cfg["ep2"]),
    "--lr",     str(cfg["lr"]),
    "--device", cfg["device"],
    "--data",   cfg["data"],
    "--out-dir",cfg["out_dir"],
]
print(">>", " ".join(cmd2))
subprocess.check_call(cmd2)
print("Done. Best checkpoints:", cfg["out_dir"])

In [ ]:
# 6. Inference smoke-test: грузит .best.pt (с фолбэком на .latest.pt), ничего не пишет на диск кроме .bsq.
sample = ROOT / "data" / "sample.wav"
if not sample.exists():
    print("Положи test-песню в data/sample.wav (любой wav/mp3) чтобы посмотреть как генерит модель.")
else:
    cmd3 = [PY, "generate.py", str(sample), "--device", cfg["device"], "--difficulty", "Expert"]
    print(">>", " ".join(cmd3))
    subprocess.check_call(cmd3)

## Что крутить (шпаргалка)

**Stage 1** (`models/stage1.py`) — реальные флаги: `--epochs`, `--steps`, `--bs`, `--lr`, `--device`, `--data`, `--out-dir`, `--resume`.

**Stage 2** (`models/stage2.py`) — реальные флаги: `--epochs`, `--lr`, `--device`, `--data`, `--out-dir`, `--resume`. **`--bs` и `--steps` нет**, батч собирается в коде. Так что ускорять Stage 2 на жирной карте в основном через `--epochs` и `--lr`.

### Если арендовал H200 (или A100 80 GB) — что делать кроме пресета

1. Открой `models/_ckpt/` как сетевую папку / монтируй на storage instance, чтобы чекпойнты не пропали при выключении.
2. Смело ставь `bs=96..128`. Дальше упираешься не в VRAM, а в то, что длинные окна (T) начинают резаться.
3. `--steps` (Stage 1) = сколько батчей на эпоху. Больше = стабильнее оценка F1, но эпоха дольше. На H200 ставь 150–200.
4. `--lr` для Adam: до ~2e-3 держит, выше лосс расходится.
5. Если есть время (а за аренду платишь) — увеличь `--epochs` до 80–100 для обоих стадий. Потолок F1 ~0.56 на 326 OST, выше пока не прыгнуть без BeatSaver.
6. Не забудь `--resume models/_ckpt/stage1.best.pt` если стейдж1 уже обучен — иначе стартанёт с нуля.
7. Early stop в Stage 1: терпение 3 эвала (3×5=15 эпох). На H200 эпохи короче — терпение по факту срабатывает быстрее, нормально.

### Если вдруг CPU

Двинь ползунок VRAM в 0 и AMP в `off` (или просто CPU) — формула выдаст bs=4, steps=40, epochs=20. Это смок на пайплайне, не полное обучение. На CPU считать — дни.